In [ ]:
import os
from googleapiclient.discovery import build

# API 키를 여기에 입력하세요
API_KEY = os.environ["YOUTUBE_API_KEY"]
# 검색할 채널의 ID를 여기에 입력하세요
# SEARCH_QUERIES = ['soccer']
SEARCH_QUERIES = ['FIFA match highlights', 'MLB highlights', "NBA highlights",  "Wimbledon highlights"]



In [ ]:
from googleapiclient.discovery import build

# YouTube API 클라이언트 생성
youtube = build('youtube', 'v3', developerKey=API_KEY)


In [ ]:
def search_videos(youtube, search_query, max_results=1000):
    video_ids = []
    next_page_token = None
    results_per_page = 50  # YouTube API max results per page

    while len(video_ids) < max_results:
        try:
            search_response = youtube.search().list(
                q=search_query,
                part='id',
                type='video',
                maxResults=results_per_page,
                pageToken=next_page_token
            ).execute()
            
            # 각 페이지의 응답 결과 확인
            ids_in_page = [item['id']['videoId'] for item in search_response['items']]
            video_ids.extend(ids_in_page)
            
            # 다음 페이지 토큰 설정
            next_page_token = search_response.get('nextPageToken')
            if not next_page_token:
                break

            # print(f"Fetched {len(ids_in_page)} videos in this page, Total fetched so far: {len(video_ids)}")
            # print(f"Next page token: {next_page_token}")

        except Exception as e:
            print(f"An error occurred: {e}")
            break
    
    return video_ids[:max_results]

# 각 검색어에 대해 영상 검색 및 ID 리스트 반환
all_video_ids = {}
# SEARCH_QUERIES = ["soccer"]  # Define your search queries here

for query in SEARCH_QUERIES:
    print(f"Searching for query: {query}")
    video_ids = search_videos(youtube, query)
    all_video_ids[query] = video_ids
    print(f"Video number for '{query}': {len(video_ids)}")

print(f"All video IDs: {all_video_ids}")


In [ ]:
import json

def save_to_json(data, filename):
    with open(filename, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)

# 검색 결과를 저장할 JSON 파일명
output_filename = 'video_ids.json'

# 검색어와 영상 ID 리스트를 JSON 파일로 저장
save_to_json(all_video_ids, output_filename)

print(f"Video IDs have been saved to {output_filename}")


In [ ]:
import isodate

def convert_duration(duration):
    parsed_duration = isodate.parse_duration(duration)
    return int(parsed_duration.total_seconds())


In [ ]:
def get_video_lengths(youtube, video_ids):
    video_lengths = []
    
    for i in range(0, len(video_ids), 50):  # 50개씩 나눠서 요청
        video_response = youtube.videos().list(
            id=','.join(video_ids[i:i+50]),
            part='contentDetails'
        ).execute()
        
        # 영상 길이를 초 단위로 변환하여 리스트에 저장
        for item in video_response['items']:
            duration = item['contentDetails']['duration']
            video_length = convert_duration(duration)
            video_lengths.append(video_length)
    
    return video_lengths

# 각 검색어에 대해 영상 길이 가져오기
all_video_lengths = {}
for query, video_ids in all_video_ids.items():
    video_lengths = get_video_lengths(youtube, video_ids)
    all_video_lengths[query] = video_lengths
    print(f"Video lengths for '{query}' in seconds:", video_lengths)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy.signal import find_peaks

plt.figure(figsize=(8, 5))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']  # 색상 리스트

for i, (query, video_lengths) in enumerate(all_video_lengths.items()):
    filtered_video_lengths = [length for length in video_lengths if -200 <= length <= 800]
    
    kde = stats.gaussian_kde(filtered_video_lengths)
    x_range = np.linspace(-200, 800, 1000)
    y_range = kde(x_range)
    
    # 그래프 그리기와 내부 색칠하기
    plt.fill_between(x_range, y_range, alpha=0.3, color=colors[i], 
                     label=f"{query}({len(video_lengths)})")
    
    ax.plot(x_range, y_range, color=colors[i], linewidth=2, 
            label=f"{query}({len(video_lengths)})")
    # 피크 찾기
    peaks, _ = find_peaks(y_range, height=0, distance=50, prominence=0.0001)
    
    # 피크 표시
    plt.plot(x_range[peaks], y_range[peaks], "o", color=colors[i])
    
    # 피크 레이블 표시
    for px, py in zip(x_range[peaks], y_range[peaks]):
        plt.annotate(f'{px:.1f}s', (px, py), xytext=(0, 5), 
                     textcoords='offset points', ha='center', va='bottom',
                     fontsize=10, color=colors[i])

plt.title('Density Plot of Highlight Video', fontsize=16)
plt.xlabel('Length (seconds)', fontsize=14)
plt.ylabel('Density', fontsize=14)

# 범례 스타일 조정
plt.legend(fontsize=12, loc='upper right', frameon=True, 
           fancybox=True, framealpha=0.8, ncol=1)

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

plt.figure(figsize=(10, 6))

for query, video_lengths in all_video_lengths.items():
    filtered_video_lengths = [length for length in video_lengths if 0 <= length <= 1000]
    
    kde = stats.gaussian_kde(filtered_video_lengths)
    x_range = np.linspace(0, 1000, 1000)
    plt.plot(x_range, kde(x_range), label=query)

plt.title('Smooth Density Plot of Video Lengths')
plt.xlabel('Length (seconds)')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

for query, video_lengths in all_video_lengths.items():
    filtered_video_lengths = [length for length in video_lengths if 0 <= length <= 1000]
    sns.kdeplot(filtered_video_lengths, label=query, bw_adjust=0.5)

plt.title('Smooth Density Plot of Video Lengths')
plt.xlabel('Length (seconds)')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 각 검색어에 대해 0에서 1000초 사이의 영상 길이만 포함한 히스토그램과 밀도 추정 곡선 그리기
for query, video_lengths in all_video_lengths.items():
    filtered_video_lengths = [length for length in video_lengths if 0 <= length <= 1000]
    
    plt.figure(figsize=(10, 6))
    
    # 히스토그램 그리기
    count, bins, ignored = plt.hist(filtered_video_lengths, bins=20, alpha=0.5, label=f'{query} histogram', density=True, edgecolor='black')
    
    # 밀도 추정 곡선 그리기
    density = np.histogram(filtered_video_lengths, bins=bins, density=True)[0]
    centers = 0.5 * (bins[1:] + bins[:-1])
    
    # x축 범위를 0에서 1000초로 설정
    plt.xlim(0, 1000)

    # 그래프 꾸미기
    total_videos = len(video_lengths)
    filtered_videos = len(filtered_video_lengths)
    plt.title(f'Video Lengths for "{query}" (0-1000 seconds)\nTotal: {total_videos}, Filtered: {filtered_videos}')
    plt.xlabel('Length (seconds)')
    plt.ylabel('Density')
    plt.legend()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 각 검색어에 대해 0에서 1000초 사이의 영상 길이만 포함한 히스토그램과 밀도 추정 곡선 그리기
for query, video_lengths in all_video_lengths.items():
    filtered_video_lengths = [length for length in video_lengths if 0 <= length <= 1000]
    
    plt.figure(figsize=(10, 6))
    
    # 히스토그램 그리기
    count, bins, ignored = plt.hist(filtered_video_lengths, bins=20, alpha=0.5, label=f'{query} histogram', density=True, edgecolor='black')
    
    # 밀도 추정 곡선 그리기
    density = np.histogram(filtered_video_lengths, bins=bins, density=True)[0]
    centers = 0.5 * (bins[1:] + bins[:-1])
    # plt.plot(centers, density, '--', label=f'{query} density')

    # x축 범위를 0에서 1000초로 설정
    plt.xlim(0, 1000)

    # 그래프 꾸미기
    plt.title(f'Histogram and Density Plot of Video Lengths for "{query}" (0-1000 seconds)')
    plt.xlabel('Length (seconds)')
    plt.ylabel('Density')
    plt.legend()
    plt.show()
